In [2]:
from spin_lattices import SpinLattice, KagomeLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
from vmc_amplitude import apply_off_diag_to_basis_states, apply_diag_to_basis_states
from scipy.sparse import csr_matrix, diags, eye
import numpy as np
from scipy.sparse.linalg import eigsh

2023-08-16 20:57:49.994 | DEBUG    | lattice_symmetries:__init__:50 - Initializing Haskell runtime...
2023-08-16 20:57:49.997 | DEBUG    | lattice_symmetries:__init__:52 - Initializing Chapel runtime...
2023-08-16 20:57:49.997 | DEBUG    | lattice_symmetries:__init__:52 - Initializing Chapel runtime...
2023-08-16 20:57:50.047 | DEBUG    | lattice_symmetries:__init__:54 - Setting Python exception handler...
set_python_exception_handler ...


In [3]:
lattice = KagomeLattice(2, 3)
system = HeisenbergJ1J2(lattice, 1, 1, use_symmetries=False, spin_inversion=None)
system.get_eigenstates(1)

2023-08-16 20:57:52.556 | DEBUG    | heisenberg_hamiltonians:__init__:482 - number_spins=18
2023-08-16 20:57:52.558 | DEBUG    | heisenberg_hamiltonians:__init__:492 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-16 20:57:52.571 | DEBUG    | heisenberg_hamiltonians:__init__:501 - Hilbert space dimension is 48620
2023-08-16 20:57:52.582 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:69 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x3-1.0-1.0-False-None-1.pickle
2023-08-16 20:57:52.585 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:121 - Ground state energy is -32.1930830942


(array([-32.19308309]),
 array([[ 4.67734071e-06],
        [ 4.05907513e-05],
        [-1.98695729e-05],
        ...,
        [ 1.98695729e-05],
        [-4.05907513e-05],
        [-4.67734071e-06]]))

In [4]:
def get_csr_hamiltonian(system: SpinSystem):
    states, coeffs, row_idxs = apply_off_diag_to_basis_states(
        system.hamiltonian, system.basis.states
    )
    columns = system.basis.index(states)
    upper_triangular = csr_matrix(
        (coeffs, columns, row_idxs),
        shape=(system.basis.states.shape[0], system.basis.states.shape[0]),
    )
    diagonal = diags(
        apply_diag_to_basis_states(system.hamiltonian, system.basis.states)
    )
    return (upper_triangular + diagonal).real

In [5]:
H = get_csr_hamiltonian(system)

In [6]:
eigsh(H, which="SA", k=1)

(array([-32.19308309]),
 array([[ 4.67734071e-06],
        [ 4.05907513e-05],
        [-1.98695729e-05],
        ...,
        [ 1.98695729e-05],
        [-4.05907513e-05],
        [-4.67734071e-06]]))

In [7]:
ground_state = system.get_ground_state_coeffs(system.basis.states)
signs = np.sign(ground_state)
H_corrected = diags(signs) @ H @ diags(signs)

In [8]:
def overlap(x, y):
    return x @ y / (np.linalg.norm(x) * np.linalg.norm(y))

In [19]:
signs = np.sign(ground_state)
H_corrected = diags(signs) @ H @ diags(signs)
H_corrected_shifted = 50 * eye(H_corrected.shape[0]) - H_corrected
x = np.random.uniform(0, 1, size=H_corrected.shape[0])
noise_level = 0.5
for i in range(200):
    noise = np.random.uniform(-noise_level, noise_level, size=x.shape[0])
    x = H_corrected_shifted @ (x * (1 + noise))
    x = x / np.linalg.norm(x)
    print(i, overlap(x, np.abs(ground_state)))

0 0.4988652390592742
1 0.5936594398968662
2 0.6717501577729265
3 0.734887779481719
4 0.7821856397758166
5 0.8157121455674939
6 0.8405660573992813
7 0.8588990408737246
8 0.873378571712788
9 0.8831910970264099
10 0.8905048677233193
11 0.9005697091282371
12 0.9041836703206018
13 0.9066088032532172
14 0.9085623815751402
15 0.9106016812248958
16 0.9136341299018893
17 0.9122724804621073
18 0.9148935169180012
19 0.9124475699833142
20 0.9093227056866966
21 0.9089169331013038
22 0.9102948368889915
23 0.9112396007054937
24 0.91401523935982
25 0.9114792326053675
26 0.9120655838452731
27 0.9102271720008197
28 0.9063233134996835
29 0.9075296472436007
30 0.9096965860436823
31 0.90958060619698
32 0.9113914715140375
33 0.9093500478638329
34 0.9082212532168878
35 0.9073078664985452
36 0.9093332980488958
37 0.9086130648783924
38 0.9060762407623771
39 0.9085460109463676
40 0.9100240809538895
41 0.9085607173467332
42 0.9070239316233647
43 0.90988756935337
44 0.9082244100731055
45 0.9074568216571921
46 0.9

In [10]:
overlap(H_corrected_shifted @ x, x)

0.9999999999972491

In [32]:
H_corrected_shifted @ x / x

array([82.19235526, 82.19332478, 82.19235441, ..., 82.19235441,
       82.19332478, 82.19235526])

In [25]:
system.get_eigenstates(1)

(array([-32.19308309]),
 array([[ 4.67734071e-06],
        [ 4.05907513e-05],
        [-1.98695729e-05],
        ...,
        [ 1.98695729e-05],
        [-4.05907513e-05],
        [-4.67734071e-06]]))

In [19]:
system.get_eigenstates(1)[1][:, 0]

array([ 4.67734071e-06,  4.05907513e-05, -1.98695729e-05, ...,
        1.98695729e-05, -4.05907513e-05, -4.67734071e-06])

In [18]:
eigsh(H_corrected, which="SA")[1][:, 0]

array([4.67734071e-06, 4.05907513e-05, 1.98695729e-05, ...,
       1.98695729e-05, 4.05907513e-05, 4.67734071e-06])

In [37]:
eigsh(H_corrected_shifted, k=1, which="LM")[1][:, 0]

array([4.67734071e-06, 4.05907513e-05, 1.98695729e-05, ...,
       1.98695729e-05, 4.05907513e-05, 4.67734071e-06])